In [3]:
# Local path config

import sys
from pathlib import Path

current_dir = Path.cwd()

root_dir = current_dir
while not (root_dir / 'utils').is_dir() and root_dir != root_dir.parent:
    root_dir = root_dir.parent

if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

# Decision Tree Regressor - From scratch and `scikit-learn` approach

The main mechanism of Decision Tree Regressor:

1. Starting with the entire training dataset at root node
2. Choose a feature that minimizes a specific error metrics (such as `MAE`) to split the data into sub-datasets.
3. Create child node based on the split, where each child represents a subset of the data aligned with the corresponding feature values.
4. Repeat `Step 2`, `Step 3` for each child node, continuing splitting the data until a stop condition is met.
5. Assign a final predicted value to each leaf node, typically **average values of the target values** in that node.

## From Scratch approach

Import the libraries, load the train dataset

In [ ]:
from __future__ import annotations

import pandas as pd 
import numpy as np
import joblib
import sklearn.tree
from dataclasses import dataclass
from utils.custom_hyperparameter_tuning import CustomGridSearchCV
from utils.custom_cv import CustomKFold

X_train = joblib.load('./data/ready_for_train/X_train.pkl')
X_test = joblib.load('./data/ready_for_train/X_test.pkl')

y_train = joblib.load('./data/ready_for_train/y_train.pkl')
y_test = joblib.load('./data/ready_for_train/y_test.pkl')

# Previewing Train, test shapes

print('Training dataset shape:')
print(f'X: {X_train.shape}')
print(f'y: {y_train.shape}')

print('Testing dataset shape:')
print(f'X: {X_test.shape}')
print(f'y: {y_test.shape}')


Training dataset shape:
X: (78789, 44)
y: (78789,)
Testing dataset shape:
X: (19698, 44)
y: (19698,)


Implementating Decision Tree Regressor from scratch

In [9]:
@dataclass
class Node:
    feature: str | float | None = None
    threshold: float | None = None
    left: Node | None = None
    right: Node | None = None
    value: float | None = None

    def is_leaf_node(self):
        return self.value is not None

class DecisionTreeRegressor:
    def __init__(self, max_depth=5, min_samples_split=3, min_samples_leaf=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.root = None

    def __mse(self, y_actual: np.ndarray | pd.Series, y_hat: float):
        """
        Mean Square Error = 1/n * (y - y_hat)^2
        """
        n = len(y_actual)
        r = y_actual - y_hat
        r = r ** 2
        r = np.sum(r)
        return r / n

    def set_params(self, **params):
        for key, value in params.items():
            setattr(self, key, value)
        return self

    def get_params(self):
        return {
        "max_depth": self.max_depth,
        "min_samples_split": self.min_samples_split,
        "min_samples_leaf": self.min_samples_leaf
    }

    def fit(self, X, y):
        self.root = self.__build_tree(X, y, 0)
        return self

    def predict(self, X: pd.DataFrame):
        X_dicts = X.to_dict(orient='records')
        return np.array([self.__traverse_tree(x, self.root) for x in X_dicts])

    def __traverse_tree(self, x: dict, node: Node):
        if node.is_leaf_node():
            return node.value

        if x[node.feature] <= node.threshold:
            return self.__traverse_tree(x, node.left)
        
        return self.__traverse_tree(x, node.right)

    def __build_tree(self, X: pd.DataFrame, y: pd.Series, depth: int):
        n_samples = X.shape[0]

        df = X.copy()
        df['Y'] = y


        # Stop condition:
            # 1. Reaching maximum depth
            # 2. Current samples splitted in a node <= minimum samples splitting

        if (depth >= self.max_depth 
            or n_samples < self.min_samples_split
            or n_samples < 2*self.min_samples_leaf
            or y.nunique() == 1
            ):

            value = y.mean()
            return Node(value=value)
        
        # Getting the best feature, threshold and split the dataset based on threshold
        best_feature, best_threshold = self._get_best_split_criteria(X, y)

        if best_feature is None:
            value = y.mean()
            return Node(value=value)

        left_mask = X[best_feature] <= best_threshold

        X_left = X[left_mask]
        y_left = y[left_mask]

        X_right = X[~left_mask]
        y_right = y[~left_mask]

        # Recursively build the left and right branches using new dataset
        left_child = self.__build_tree(X_left, y_left, depth + 1)
        right_child = self.__build_tree(X_right, y_right, depth + 1)

        return Node(feature=best_feature,
                    threshold=best_threshold,
                    left=left_child,
                    right=right_child,
                    value=None)

    def _get_best_split_criteria(self, X: pd.DataFrame, y: pd.Series):
        """
        We will use midpoints to calculate all possible split points for all features of the dataset
        """
        current_mse = self.__mse(y, y.mean())
        best_feature = None
        best_threshold = None

        # Get X_train features 
        features = X.columns
        y = y.values
            
        for feature in features:
            # 2. Convert the current feature to a NumPy array
            x_arr = X[feature].values
            
            # 3. Get sorted unique values and calculate midpoints
            unique_vals = np.sort(np.unique(x_arr))
            splits = (unique_vals[:-1] + unique_vals[1:]) / 2.0
            
            for split in splits:
                # 4. NumPy boolean masking (lightning fast)
                left_mask = x_arr <= split
                
                n_left = np.sum(left_mask)
                n_right = len(y) - n_left

                # Check bounds
                if n_left < self.min_samples_leaf or n_right < self.min_samples_leaf:
                    continue
                
                # 5. Apply the boolean mask directly to the y array
                y_left = y[left_mask]
                y_right = y[~left_mask]
                
                # Calculate MSE for each side
                left_mse = self.__mse(y_left, y_left.mean())
                right_mse = self.__mse(y_right, y_right.mean())
                
                # Calculate weighted MSE
                weight_mse = ((left_mse * n_left) + (n_right * right_mse)) / len(y)

                # Applying the best split condition: Weighted MSE of split point is the lowest
                if weight_mse < current_mse:
                    current_mse = weight_mse
                    best_threshold = split
                    best_feature = feature

        return (best_feature, best_threshold)    


## `sklearn-learn` library approach

In [17]:
re = sklearn.tree.DecisionTreeRegressor(max_depth = 8, min_samples_split=100, min_samples_leaf=500)
re.fit(X_train, y_train)


,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.For an example of how ``max_depth`` influences the model, see:ref:`sphx_glr_auto_examples_tree_plot_tree_regression.py`.",8
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",100
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",500
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 0.24 Poisson deviance criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",None
,"max_leaf_nod

## Hyperparameters Tuning

### Set-up Decision Tree Regressor's hyperparameter grid

In [15]:
dtr_grid = {'max_depth': [3,5,8,12],
            'min_samples_split': [100,500,1000],
            'min_samples_leaf': [50,100,500]}

In [16]:
cv = CustomKFold(n_splits=3, shuffle=True, random_state=42)
dtr = DecisionTreeRegressor()

dtr_grid_search = CustomGridSearchCV(estimator=dtr,param_grid = dtr_grid,cv=cv,scoring = ['r2','neg_mse','neg_rmse'])
dtr_grid_search.fit(X_train, y_train)


Bắt đầu GridSearchCV: 36 tổ hợp tham số, 3 folds.
[1/36] Params: {'max_depth': 3, 'min_samples_split': 100, 'min_samples_leaf': 50} --> r2: 0.4794, neg_mse: -9219491.3027, neg_rmse: -3036.3612
[2/36] Params: {'max_depth': 3, 'min_samples_split': 100, 'min_samples_leaf': 100} --> r2: 0.4791, neg_mse: -9224416.4528, neg_rmse: -3037.1724
[3/36] Params: {'max_depth': 3, 'min_samples_split': 100, 'min_samples_leaf': 500} --> r2: 0.4800, neg_mse: -9208834.8158, neg_rmse: -3034.6058
[4/36] Params: {'max_depth': 3, 'min_samples_split': 500, 'min_samples_leaf': 50} --> r2: 0.4793, neg_mse: -9221141.3031, neg_rmse: -3036.6328
[5/36] Params: {'max_depth': 3, 'min_samples_split': 500, 'min_samples_leaf': 100} --> r2: 0.4790, neg_mse: -9226066.4533, neg_rmse: -3037.4440
[6/36] Params: {'max_depth': 3, 'min_samples_split': 500, 'min_samples_leaf': 500} --> r2: 0.4800, neg_mse: -9208834.8158, neg_rmse: -3034.6058
[7/36] Params: {'max_depth': 3, 'min_samples_split': 1000, 'min_samples_leaf': 50} --> r